# Movie Genre Prediction: Model Training & Tracking

Notebook ini mendemonstrasikan proses *End-to-End* pelatihan model Machine Learning menggunakan algoritma **LinearSVC** dan pembobotan kata **TF-IDF**, sekaligus melacak eksperimennya secara otomatis menggunakan **MLflow**.

In [1]:
import os
import pickle
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

# Setup direktori MLflow tracking & tempat simpan model
mlflow.set_tracking_uri("file:./mlruns")
os.makedirs("models", exist_ok=True)
print("Libraries loaded and MLflow configured.")

c:\Users\Lenovo\anaconda3\envs\ktp-ocr\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded and MLflow configured.


### Data Loading & Preprocessing

In [5]:
df = pd.read_csv("D:\Movie_genre_prediction\data\movies.csv")

# Filter ke 5 genre utama agar model lebih fokus
MAIN_GENRES = ["Action", "Comedy", "Drama", "Horror", "Thriller"]
df = df[df["genre"].isin(MAIN_GENRES)].reset_index(drop=True)

print(f"Distribusi Dataset (Total: {len(df)} baris):")
print(df['genre'].value_counts())

df.head()

Distribusi Dataset (Total: 2935 baris):
genre
Drama       1011
Comedy       870
Action       654
Horror       244
Thriller     156
Name: count, dtype: int64


,overview,genre
0,"Combat has taken its toll on Rambo, but he's f...",Action
1,"Due to a curse from his former master Profion,...",Action
2,"On a fall night in 2003, Harvard undergrad and...",Drama
3,After accidentally witnessing a mafia hit in t...,Comedy
4,When George and her colleagues get a new boss ...,Drama


### Feature Engineering (TF-IDF) & Data Splitting

In [7]:
X = df["overview"]
y = df["genre"]

# Split 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2,
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Fitur berhasil diekstrak. Ukuran X_train: {X_train_vec.shape}, Ukuran X_train: {X_test_vec.shape}")

Fitur berhasil diekstrak. Ukuran X_train: (2348, 10000), Ukuran X_train: (587, 10000)


In [9]:
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("mlflow").setLevel(logging.ERROR)

# Setelah itu baru jalankan train() atau kode MLflow Anda


### MLflow: Model Training, Evaluation & Auto-Logging

In [ ]:
mlflow.set_experiment("movie-genre-prediction")

with mlflow.start_run() as run:
    # 1. Inisiasi dan Train Model
    model = LinearSVC(max_iter=2000, C=5.0, dual="auto")
    model.fit(X_train_vec, y_train)
    
    # 2. Pengujian (Testing)
    y_pred = model.predict(X_test_vec)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    
    # 3. Logging Parameter Eksperimen ke MLflow
    mlflow.log_param("model", "LinearSVC")
    mlflow.log_param("max_features", 10000)
    mlflow.log_param("ngram_range", "(1,2)")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("C", 5.0)
    
    # 4. Logging Metrics
    mlflow.log_metric("accuracy", accuracy)
    
    # 5. Sinkronisasi Model & Artifact ke Repository
    mlflow.sklearn.log_model(model, "model")
    pickle.dump(vectorizer, open("models/vectorizer.pkl", "wb"))
    pickle.dump(model, open("models/model.pkl", "wb"))
    
    print("====================================================")
    print(f"\u2728 Training Selesai! Skor Akurasi: {accuracy:.4f}")
    print("====================================================")
    print("\n\ud83d\udcca Classification Report:\n")
    print(report)
    print(f"\nMLflow Run ID: {run.info.run_id}")
    print("\ud83d\udd17 Model & Vectorizer telah disimpan.")
